In [ ]:
using ProgressBars, ChaosTools
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
ω_b = 7.65*2*pi#(7+rand(1)[1])*2pi
ω_a = ω_b / 2

φ_a = 0.11
φ_b = 0.204

E_J = 37.12*2*pi


##Drive
ϵ_p = 0.2#5pi
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b = 1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0.01;#3e-3*κ_2 #from Leon's paper

κ_a_tilde, κ_b_tilde = κ_a/ω_b, κ_b/ω_b
β_a, β_b = 4E_J*sin(ϵ_p)φ_a^2/(ω_b/2), 4E_J*sin(ϵ_p)φ_b^2/ω_b
λ_a, λ_b = φ_a, φ_b
ϵ_d_p = ϵ_d*sqrt(2)/ω_b
ν_d = 1;

In [ ]:
println("β_a="*string(β_a))
println("β_b="*string(β_b))
println("λ_a="*string(λ_a))
println("λ_b="*string(λ_b))

In [ ]:
t_max = 30/κ_b*ω_b
t_min = 0.0
tspan = (t_min, t_max)
re_a_0, im_a_0 = 1, 1
x_a_0, p_a_0=sqrt(2)*re_a_0, sqrt(2)*im_a_0 *(1/2)
x_b_0, p_b_0= 4.0, 4.0

system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
drive = DriveOneMode(ϵ_d_p, ν_d, buffer_phase)
environment = MeanField(κ_a_tilde, κ_b_tilde)

system_prepared = prepare(system)
environment_prepared = prepare(system, environment)
drive_prepared = prepare(drive)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

prob = ODEProblem(equation_1ODE!, u_0_new, tspan, p)
sol = solve(prob, Tsit5(),dt=dt)
x_a = getindex.(sol.u,1)
x_b = getindex.(sol.u,2)
p_a = getindex.(sol.u,3)
p_b = getindex.(sol.u,4)
t_list = sol.t

#x_a, x_b, p_a, p_b, t_list = default_solver(u_0_new, tspan, p, dt, equation=equation_1ODE!)
a = @. (x_a+1im*p_a)/sqrt(2)
a_rot = a.*exp.(1im*t_list/2)

b = @. (x_b+1im*p_b)/sqrt(2)
b_rot = b.*exp.(1im*t_list);


t_w = 30/κ_b_tilde
problem = CoupledODEs(equation_1ODE!, u_0_new, p)
lyapunov(problem, 10/κ_b_tilde, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)

In [ ]:
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))
plot!(xlabel = L"\Re(a)", ylabel= L"\Im(a)")

In [ ]:
plot(real.(b_rot[1:end]), imag.(b_rot[1:end]))
plot!(xlabel = L"\Re(b)", ylabel= L"\Im(b)")

In [ ]:
F, freqs = FFT_perso(x_b, t_list)
power_density = abs.(F).^2
power_density_normalized = power_density ./sum(power_density)
Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))

In [ ]:
t_max=10/κ_b_tilde
re_a_0, im_a_0 = 0, 0
x_a_0, p_a_0=sqrt(2)*re_a_0, sqrt(2)*im_a_0 *(1/2)
x_b_0, p_b_0=0.0, 0.0

system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
drive = DriveOneMode(ϵ_d_p, ν_d, buffer_phase)
environment = MeanField(κ_a_tilde, κ_b_tilde)

system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
κ_a_list = 10.0 .^(-9:.2:4)
ϵ_p_list = 0.192:0.05:1.214

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(κ_a_list), length(ϵ_p_list))
t_w = 30/κ_b_tilde
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    β_a, β_b = 4E_J*sin(ϵ_p_i)φ_a^2/(ω_b/2), 4E_J*sin(ϵ_p_i)φ_b^2/ω_b
    system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
    system_prepared = prepare(system)
    for j in 1:length(κ_a_list)
        κ_a_tilde_j = κ_a_list[j]/ω_b
        environment = MeanField(κ_a_tilde_j, κ_b_tilde)
        environment_prepared = prepare(system,environment)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,log10.(κ_a_list),Lyap_coef_chaos)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\log_{10}(\kappa_a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(log10.(κ_a_list)),maximum(log10.(κ_a_list))),
    colorbar_title="\nMax Lyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max/ω_b))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w/ω_b))",
    clims=(minimum(Lyap_coef_chaos),maximum(Lyap_coef_chaos)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);
heatmap(g2_list/2/pi*1e3,log10.(κ_a_list),Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\log_{10}(\kappa_a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(log10.(κ_a_list)),maximum(log10.(κ_a_list))),
    colorbar_title="\nMax Lyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max/ω_b))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w/ω_b))",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)

# Shannon entropy

In [ ]:
t_max=30/κ_b_tilde

re_a_0, im_a_0 = 0, 0
x_a_0, p_a_0=sqrt(2)*re_a_0, sqrt(2)*im_a_0 *(1/2)
x_b_0, p_b_0=0.0, 0.0

system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
drive = DriveOneMode(ϵ_d_p, ν_d, buffer_phase)
environment = MeanField(κ_a_tilde, κ_b_tilde);

In [ ]:
κ_a_list = 10.0 .^(-9:.2:2)
ϵ_p_list = 0.02:0.05:1.214

In [ ]:
Shannon_entropy_matrix = Array{Float64}(undef, length(κ_a_list), length(ϵ_p_list))
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    β_a, β_b = 4E_J*sin(ϵ_p_i)φ_a^2/(ω_b/2), 4E_J*sin(ϵ_p_i)φ_b^2/ω_b
    system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
    for j in 1:length(κ_a_list)
        κ_a_tilde_j = κ_a_list[j]/ω_b
        environment = MeanField(κ_a_tilde_j, κ_b_tilde)
        _, xb, t_list = simulator(t_max, system, drive, environment)
        F, freqs = FFT_perso(real.(xb), t_list)
        power_density = abs.(F).^2
        power_density_normalized = power_density ./sum(power_density)
        Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
        Shannon_entropy_matrix[j,i] = Shannon_entropy
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,log10.(κ_a_list), Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\log_{10}(\kappa_a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(log10.(κ_a_list)),maximum(log10.(κ_a_list))),
    colorbar_title="Shannon entropy", title = "b-mode "*L"κ_b t= "*"$(round(κ_b*t_max/ω_b))",
    clims=(minimum(Shannon_entropy_matrix),maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../figures/heatmap_shannon_entropy_kappa_a_sweep_19_12_2025.pdf")

In [ ]:
t_max=90/κ_b_tilde

re_a_0, im_a_0 = 0, 0
x_a_0, p_a_0=sqrt(2)*re_a_0, sqrt(2)*im_a_0 *(1/2)
x_b_0, p_b_0=0.0, 0.0

system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
drive = DriveOneMode(ϵ_d_p, ν_d, buffer_phase)
environment = MeanField(κ_a_tilde, κ_b_tilde);

In [ ]:
φ_a_list = 10.0 .^(-3:.1:-0.5)
ϵ_p_list = 0.192:0.05:1.214

In [ ]:
Shannon_entropy_matrix_lambda = Array{Float64}(undef, length(φ_a_list), length(ϵ_p_list))
t_w = 30/κ_b_tilde
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    for j in 1:length(φ_a_list)
        φ_a = φ_a_list[j]
        β_a, β_b = 4E_J*sin(ϵ_p_i)φ_a^2/(ω_b/2), 4E_J*sin(ϵ_p_i)φ_b^2/ω_b
        λ_a = φ_a
        system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
        system_prepared = prepare(system)
        environment_prepared = prepare(system, environment)
        drive_prepared = prepare(drive)

        dt = get_dt(system, environment)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0);

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        prob = ODEProblem(equation_1ODE!, u_0_new, tspan, p)
        sol = solve(prob, Tsit5(),dt=dt)
        x_a = getindex.(sol.u,1)
        F, freqs = FFT_perso(x_a, t_list)
        power_density = abs.(F).^2
        power_density_normalized = power_density ./sum(power_density)
        Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
        Shannon_entropy_matrix_lambda[j,i] = Shannon_entropy
    end
end

In [ ]:
heatmap(ϵ_p_list, log10.(φ_a_list), Shannon_entropy_matrix_lambda)
plot!(xlabel=L"\varepsilon_p", ylabel=L"\log_{10}(\varphi_a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(ϵ_p_list),maximum(ϵ_p_list)),ylims=(minimum(log10.(φ_a_list)),maximum(log10.(φ_a_list))),
    colorbar_title="\n Shannon entropy", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max/ω_b))",
    clims=(minimum(Shannon_entropy_matrix_lambda),maximum(Shannon_entropy_matrix_lambda)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")